## [内置中间件](https://docs.langchain.com/oss/python/langchain/middleware/built-in)

#### 加载环境

In [ ]:
from dotenv import load_dotenv

# 读取 .env 配置
load_dotenv(override=True)

#### 初始化

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model='deepseek:deepseek-flash'
)

#### Summarization

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model='deepseek:deepseek-flash',
    middleware=[SummarizationMiddleware(
        model=model,
        # 或关系
        trigger=[("tokens", 20), ("messages", 5)],
        # 且关系
        # trigger={"tokens": 20, "messages": 5},
        # 或和且关系
        # trigger=[
        #     {"tokens": 100, "messages": 3},
        #     {"tokens": 20, "messages": 5},
        # ],
        keep=("messages", 2)
    )]
)

messages = [
    SystemMessage("你是耐心的 AI 助手"),
    HumanMessage("你好"),
    AIMessage("你好，请问有什么可以帮你"),
    HumanMessage("帮我查询北京的天气"),
    AIMessage("北京今天天气晴朗～"),
    HumanMessage("谢谢"),
]
resp = agent.invoke({'messages': messages})
for msg in resp["messages"]:
    msg.pretty_print()

#### Human-in-the-loop

In [ ]:
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from rich.pretty import pprint

def read_email_tool(email_id: str) -> str:
    """通过邮件 ID 读取邮件内容"""
    return f"邮件内容: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """发送邮件"""
    return f"发送邮件给 {recipient}， 标题：'{subject}'，内容：'{body}'"

agent = create_agent(
    model='deepseek:deepseek-flash',
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "read_email_tool": False,
            "send_email_tool": {
                "allowed_decisions": ["approve", "edit", "reject"],
            },
        }
    )]
)

config: RunnableConfig = {"configurable": {"thread_id": "session01"}}
messages = [
    HumanMessage("帮我查询 ID 为 'e123456' 的邮件内容，并给 123456789@qq.com 发送标题为'测试'，内容为'你好'的邮件"),
]
resp = agent.invoke({'messages': messages}, config=config)
pprint(resp)

In [ ]:
from langgraph.types import Command

send_email_decision = {
    "type": "approve"
}

decisions = {
    "decisions": [send_email_decision]
}

resp = agent.invoke(Command(resume=decisions), config=config)
for msg in resp["messages"]:
    msg.pretty_print()

##### PII detection

In [ ]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model='deepseek:deepseek-flash',
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask"),
        PIIMiddleware("url", strategy="redact"),
        PIIMiddleware("ip", strategy="hash"),
    ]
)

messages = [
    HumanMessage("""
    帮我向 123456789@qq.com 发送一份测试邮件
    查询银行卡 5105-1051-0510-5100 的余额
    访问 https://google.com
    确认 IP 地址是否正确 192.168.0.1
    """),
]
resp = agent.invoke({'messages': messages})
for msg in resp["messages"]:
    msg.pretty_print()

##### Custom PII types

In [ ]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
import re

def detect_phone_number(content: str):
    return [
        {
            "text": m.group(0),
            "start": m.start(),
            "end": m.end(),
        } for m in re.finditer(r"[0-9]{11}", content)
    ]

agent = create_agent(
    model='deepseek:deepseek-flash',
    middleware=[
        PIIMiddleware("api_key", detector=r"sk-[a-zA-Z0-9]+", strategy="hash"),
        PIIMiddleware("detect_phone_number", detector=detect_phone_number, strategy="mask"),
    ]
)

messages = [
    HumanMessage("""
    这是不是有效的 API_KEY: sk-awef23AFEfaafaefa
    帮我给这个号码打电话：12312341234
    """),
]
resp = agent.invoke({'messages': messages})
for msg in resp["messages"]:
    msg.pretty_print()

#### To-do list

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain.agents.middleware import TodoListMiddleware
from rich.pretty import pprint

@tool
def add(a: int, b: int) -> int:
    """计算两个整数的和。"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """计算两个整数的乘积。"""
    return a * b

agent = create_agent(
    model='deepseek:deepseek-flash',
    tools=[add, multiply],
    middleware=[TodoListMiddleware()],
    system_prompt=(
        "这是任务清单的教学演示。"
        "请先调用 write_todos，把计算过程拆成三个步骤。"
        "计算必须使用 add 和 multiply 工具。"
        "每步拿到计算结果后，再更新完整清单中的任务状态。"
        "全部完成后，再单独用文字给出最终算式和答案。"
    ),
)

last_todos = None
last_state = {}

for state in agent.stream(
    {"messages": [HumanMessage("请计算：(12 + 18) × 5 + 20")]},
    stream_mode="values",
):
    last_state = state
    todos = state.get("todos")

    # values 返回完整状态，只在清单发生变化时打印
    if todos is not None and todos != last_todos:
        print("\n=== 当前任务清单 ===")
        pprint(todos)
        last_todos = todos

print("\n=== 最终回答 ===")
print(last_state["messages"][-1].text)

#### Model & Tool call limit

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware
from rich.pretty import pprint

@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    return f"{city}天气晴天~"

model_limiter = ModelCallLimitMiddleware(thread_limit=2, exit_behavior="end")
global_limiter = ToolCallLimitMiddleware(thread_limit=20, run_limit=10)
search_limiter = ToolCallLimitMiddleware(tool_name="get_weather", thread_limit=2, run_limit=1)

agent = create_agent(
    model='deepseek:deepseek-flash',
    checkpointer=InMemorySaver(),
    tools=[get_weather],
    middleware=[model_limiter, global_limiter, search_limiter],
    system_prompt="你是一个天气助手，回复简洁，需要注意单次执行和会话执行的工具调用限制"
)

config: RunnableConfig = {"configurable": {"thread_id": "session01"}}

# 第一次对话
# 天气工具第一次调用成功，第二次失败
resp = agent.invoke({'messages': [HumanMessage("帮忙查看北京和上海的天气")]}, config=config)
for msg in resp["messages"]:
    msg.pretty_print()

print("=" * 35, ">> 02 <<", "=" * 35)

# 第二次对话（相同会话）
# 天气工具调用到达限制
resp = agent.invoke({'messages': [HumanMessage("帮忙查看深圳的天气")]}, config=config)
for msg in resp["messages"]:
    msg.pretty_print()

print("=" * 35, ">> 03 <<", "=" * 35)

# 第三次对话（相同会话）
# 对话到达限制
resp = agent.invoke({'messages': [HumanMessage("帮忙查看香港的天气")]}, config=config)
for msg in resp["messages"]:
    msg.pretty_print()